# 05 — Dependency Graph Analysis
**Purpose:** Analyse architecture via the target dependency graph.
**Inputs:** `target_metrics.parquet`, `edge_list.parquet`, `critical_path.parquet`,
            `contributor_target_commits.parquet`, `git_commit_log.parquet` (optional)
**Outputs:** `data/intermediate/graph_analysis.parquet`,
             `data/intermediate/gephi/dependency_graph.gexf`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from buildanalysis.graphs import (
    build_dependency_graph,
    compute_centrality_metrics,
    compute_graph_summary,
    compute_layer_assignments,
    compute_transitive_deps,
    find_layer_violations,
)
from buildanalysis.git import compute_file_to_target_map, infer_team_assignments
from buildanalysis.export import export_dependency_graph
from buildanalysis.loading import BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)
tm = ds.target_metrics
el = ds.edge_list

HAS_GIT = ds.has_file("git_commit_log")

print(f"target_metrics: {tm.shape[0]:,} targets")
print(f"edge_list:      {el.shape[0]:,} edges")
print(f"Git commit log: {'available' if HAS_GIT else 'not found'}")

In [ ]:
bg = build_dependency_graph(tm, el, direct_only=True)
print(f"Graph: {bg.n_targets} targets, {bg.n_edges} direct edges")

## 1. Graph Summary

In [ ]:
summary = compute_graph_summary(bg)

print("GRAPH SUMMARY")
print("=" * 60)
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:<25s} {v:>12.4f}")
    else:
        print(f"  {k:<25s} {v:>12}")

# Context check
n = summary["n_targets"]
e = summary["n_edges"]
print(f"\nEdge-to-node ratio: {e / n:.1f}x (typical C++ projects: 2-5x)")
print(f"Density: {summary['density']:.4f} (lower = more modular)")
print(f"Max depth: {summary['max_depth']} layers")

## 2. Centrality Analysis

In [ ]:
centrality_df = compute_centrality_metrics(bg)
centrality_df = centrality_df.reset_index()

# Add target_type for display
type_map = tm.set_index("cmake_target")["target_type"].to_dict()
centrality_df["target_type"] = centrality_df["cmake_target"].map(type_map)

print(f"Centrality computed for {len(centrality_df):,} targets")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Betweenness centrality distribution
ax = axes[0]
ax.hist(centrality_df["betweenness"], bins=60, edgecolor="white", linewidth=0.3, color="steelblue")
ax.set_xlabel("Betweenness centrality")
ax.set_ylabel("Count")
ax.set_title("Betweenness Centrality Distribution")
ax.set_yscale("log")

# In-degree distribution (dependant count)
ax = axes[1]
ax.hist(centrality_df["in_degree"], bins=60, edgecolor="white", linewidth=0.3, color="coral")
ax.set_xlabel("In-degree (dependants)")
ax.set_ylabel("Count")
ax.set_title("In-Degree Distribution")
ax.set_yscale("log")

# Out-degree distribution (dependency count)
ax = axes[2]
ax.hist(centrality_df["out_degree"], bins=60, edgecolor="white", linewidth=0.3, color="teal")
ax.set_xlabel("Out-degree (dependencies)")
ax.set_ylabel("Count")
ax.set_title("Out-Degree Distribution")
ax.set_yscale("log")

fig.tight_layout()
plt.show()

In [ ]:
top_betweenness = centrality_df.sort_values("betweenness", ascending=False).head(20).copy()
top_betweenness.index = range(1, len(top_betweenness) + 1)

print("TOP 20 TARGETS BY BETWEENNESS CENTRALITY")
print("=" * 90)
print(top_betweenness[["cmake_target", "betweenness", "target_type", "in_degree", "out_degree"]].to_string())

In [ ]:
top_in = centrality_df.sort_values("in_degree", ascending=False).head(20).copy()
top_in.index = range(1, len(top_in) + 1)

print("TOP 20 TARGETS BY IN-DEGREE (most depended-upon)")
print("=" * 80)
print(top_in[["cmake_target", "in_degree", "target_type", "betweenness"]].to_string())

In [ ]:
top_out = centrality_df.sort_values("out_degree", ascending=False).head(20).copy()
top_out.index = range(1, len(top_out) + 1)

print("TOP 20 TARGETS BY OUT-DEGREE (most dependencies)")
print("=" * 80)
print(top_out[["cmake_target", "out_degree", "target_type", "betweenness"]].to_string())

## 3. Layer Analysis

In [ ]:
layer_df = compute_layer_assignments(bg)

max_depth = layer_df["layer"].max()
avg_depth = layer_df["layer"].mean()
print(f"Max depth:     {max_depth}")
print(f"Average depth: {avg_depth:.1f}")
print(f"Layer distribution:")
print(layer_df["layer"].value_counts().sort_index().to_string())

In [ ]:
layer_counts = layer_df["layer"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(layer_counts.index, layer_counts.values, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Layer")
ax.set_ylabel("Target count")
ax.set_title("Targets per Layer")
ax.set_xticks(range(0, max_depth + 1))
fig.tight_layout()
plt.show()

In [ ]:
violations_df = find_layer_violations(bg, layer_df)

print(f"Layer violations: {len(violations_df)}")
if len(violations_df) > 0:
    print(f"  Lateral: {(violations_df['violation_type'] == 'lateral').sum()}")
    print(f"  Upward:  {(violations_df['violation_type'] == 'upward').sum()}")

    # Severity: upward violations spanning more layers are worse
    violations_df["severity"] = abs(violations_df["dep_layer"] - violations_df["source_layer"])
    top_violations = violations_df.sort_values("severity", ascending=False).head(20).copy()
    top_violations.index = range(1, len(top_violations) + 1)

    print(f"\nTOP 20 VIOLATIONS BY SEVERITY")
    print("=" * 90)
    print(top_violations.to_string())
else:
    print("No layer violations found — strict layering is respected.")

## 4. Transitive Dependency Analysis

In [ ]:
transitive_df = compute_transitive_deps(bg)

print(f"Mean transitive fraction:   {transitive_df['transitive_fraction'].mean():.3f}")
print(f"Median transitive fraction: {transitive_df['transitive_fraction'].median():.3f}")
print(f"Max transitive fraction:    {transitive_df['transitive_fraction'].max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(transitive_df["transitive_fraction"], bins=60, edgecolor="white", linewidth=0.3, color="teal")
ax.axvline(transitive_df["transitive_fraction"].median(), color="red", linestyle="--",
           alpha=0.7, label=f"Median ({transitive_df['transitive_fraction'].median():.3f})")
ax.set_xlabel("Transitive fraction (of total codebase)")
ax.set_ylabel("Count")
ax.set_title("Transitive Dependency Footprint Distribution")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
top_transitive = transitive_df.sort_values("n_transitive_deps", ascending=False).head(20).copy()
top_transitive.index = range(1, len(top_transitive) + 1)

print("TOP 20 TARGETS BY TRANSITIVE DEPENDENCY COUNT")
print("=" * 80)
print(top_transitive.to_string())

In [ ]:
# Scatter: transitive_deps vs compile_time
scatter_df = transitive_df.merge(
    tm[["cmake_target", "compile_time_sum_ms"]].dropna(), on="cmake_target", how="inner"
)

if len(scatter_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(scatter_df["n_transitive_deps"], scatter_df["compile_time_sum_ms"],
              alpha=0.4, s=15, color="teal")
    ax.set_xlabel("Transitive dependency count")
    ax.set_ylabel("Compile time (ms)")
    ax.set_title("Transitive Dependencies vs Compile Time")
    fig.tight_layout()
    plt.show()

    corr = scatter_df[["n_transitive_deps", "compile_time_sum_ms"]].corr().iloc[0, 1]
    print(f"Pearson correlation: {corr:.3f}")
else:
    print("No matching data for scatter plot.")

## 5. CMake Visibility Analysis

In [ ]:
direct_edges = el[el["is_direct"]].copy()

if "cmake_visibility" in direct_edges.columns:
    vis_counts = direct_edges["cmake_visibility"].value_counts()
    print("EDGE VISIBILITY DISTRIBUTION (direct edges only)")
    print("=" * 50)
    for vis, count in vis_counts.items():
        print(f"  {vis:<20s} {count:>6,}")

    # Compute preliminary communities for cross-community analysis
    from buildanalysis.modularity import detect_communities_louvain
    prelim_communities = detect_communities_louvain(bg)
    comm_map = prelim_communities.set_index("cmake_target")["community"].to_dict()

    # PUBLIC edges that cross community boundaries
    public_edges = direct_edges[direct_edges["cmake_visibility"] == "PUBLIC"].copy()
    public_edges["src_community"] = public_edges["source_target"].map(comm_map)
    public_edges["dst_community"] = public_edges["dest_target"].map(comm_map)
    cross_community = public_edges[
        public_edges["src_community"] != public_edges["dst_community"]
    ]

    print(f"\nPUBLIC edges: {len(public_edges):,}")
    print(f"PUBLIC cross-community edges: {len(cross_community):,}")
    print(f"  (candidates for narrowing to PRIVATE)")

    if len(cross_community) > 0:
        # Rank by transitive dependant count of the dest_target
        dependant_counts = tm.set_index("cmake_target")["transitive_dependant_count"].to_dict()
        cross_community = cross_community.copy()
        cross_community["dest_dependants"] = cross_community["dest_target"].map(dependant_counts).fillna(0).astype(int)
        cross_community = cross_community.sort_values("dest_dependants", ascending=False)

        top_cross = cross_community.head(20).copy()
        top_cross.index = range(1, len(top_cross) + 1)

        print(f"\nTOP 20 PUBLIC CROSS-COMMUNITY EDGES BY IMPACT")
        print("=" * 100)
        print(top_cross[["source_target", "dest_target", "src_community",
                         "dst_community", "dest_dependants"]].to_string())
else:
    print("cmake_visibility column not present in edge_list — skipping visibility analysis.")

## 6. Team Assignments

In [ ]:
# Use existing target_ownership if available, otherwise infer from git log
try:
    team_df = ds.load_intermediate("target_ownership")
    team_col = "primary_team" if "primary_team" in team_df.columns else "top_contributor"
    print(f"Loaded target_ownership from notebook 02 (using column '{team_col}')")
except FileNotFoundError:
    if HAS_GIT:
        file_to_target = compute_file_to_target_map(ds.file_metrics)
        team_df = infer_team_assignments(ds.git_commit_log, file_to_target)
        team_col = "primary_team"
        print(f"Inferred team assignments from git log")
    else:
        team_df = pd.DataFrame({"cmake_target": tm["cmake_target"], "primary_team": "unknown"})
        team_col = "primary_team"
        print("No git data or target_ownership — all targets assigned to 'unknown'")

n_teams = team_df[team_col].nunique()
print(f"\nTeams: {n_teams}")
print(f"Targets with team assignment: {len(team_df):,}")

team_sizes = team_df[team_col].value_counts()
print(f"\nTARGETS PER TEAM")
print("=" * 50)
print(team_sizes.to_string())

In [ ]:
# Which teams own targets on the critical path?
critical_path_df = ds.load_intermediate("critical_path")
cp_targets = set(critical_path_df[critical_path_df["on_critical_path"]]["cmake_target"])

cp_teams = team_df[team_df["cmake_target"].isin(cp_targets)].copy()
cp_team_counts = cp_teams[team_col].value_counts()

print(f"Critical path targets: {len(cp_targets)}")
print(f"Teams owning critical path targets: {cp_team_counts.shape[0]}")
print(f"\nCRITICAL PATH TARGETS PER TEAM")
print("=" * 50)
print(cp_team_counts.to_string())

## 7. Gephi Export

In [ ]:
# Load critical path targets
critical_path_targets = set(
    critical_path_df[critical_path_df["on_critical_path"]]["cmake_target"]
)

# Use communities from notebook 06 if available, otherwise use preliminary Louvain
try:
    communities_df = ds.load_intermediate("communities")
    print("Loaded communities from notebook 06")
except FileNotFoundError:
    from buildanalysis.modularity import detect_communities_louvain
    communities_df = detect_communities_louvain(bg)
    print(f"Computed preliminary Louvain communities: {communities_df['community'].nunique()} communities")
    print("Note: re-export after notebook 06 for final communities.")

path = export_dependency_graph(
    bg=bg,
    centrality=centrality_df,
    layers=layer_df,
    communities=communities_df,
    timing=tm,
    team_assignments=team_df,
    critical_path_targets=critical_path_targets,
    output_path=INTERMEDIATE_DIR / "gephi" / "dependency_graph.gexf",
)
print(f"\nGEXF file: {path}")

### Gephi Usage Guide

**Setup:**
1. Open `data/intermediate/gephi/dependency_graph.gexf` in Gephi
2. Run **Yifan Hu** or **ForceAtlas2** layout (LinLog mode, scaling ~2.0, prevent overlap)
3. Size nodes by `compile_time_s`
4. Colour nodes by `community` or `team`
5. Use the Ranking panel to set edge thickness by betweenness

**What to look for:**
- **Critical path targets:** Nodes with `on_critical_path = true` — the serial
  bottleneck of the build
- **Bridge targets:** High betweenness centrality — removing or splitting these
  could decouple communities
- **Heavy transitive footprints:** Targets pulling in a large fraction of the
  codebase transitively — small changes ripple widely
- **Layer violations:** Edges marked `is_layer_violation = true` — architectural
  layering is not respected
- **Conway's law check:** Compare `community` (structural) vs `team` (organisational)
  — mismatches suggest ownership friction

In [ ]:
print("CANDIDATE NODES FOR GEPHI INVESTIGATION")
print()

# Critical path targets with times
cp_with_time = critical_path_df[critical_path_df["on_critical_path"]].sort_values(
    "build_time_ms", ascending=False
)
print("--- Critical path targets ---")
for _, row in cp_with_time.iterrows():
    print(f"  {row['cmake_target']:<50s} {row['build_time_ms']:>10,.0f} ms")

print()

# Bridge targets (high betweenness) with community and team
bridges = centrality_df.sort_values("betweenness", ascending=False).head(15).copy()
bridges = bridges.merge(
    communities_df[["cmake_target", "community"]], on="cmake_target", how="left"
)
bridges = bridges.merge(
    team_df[["cmake_target", team_col]], on="cmake_target", how="left"
)
print("--- Top 15 bridge targets (high betweenness) ---")
for _, row in bridges.iterrows():
    print(f"  {row['cmake_target']:<50s} betweenness={row['betweenness']:.4f}  "
          f"community={row.get('community', '?')}  team={row.get(team_col, '?')}")

print()

# Heaviest transitive footprints
top_trans = transitive_df.sort_values("n_transitive_deps", ascending=False).head(15)
print("--- Top 15 heaviest transitive footprints ---")
for _, row in top_trans.iterrows():
    print(f"  {row['cmake_target']:<50s} direct={row['n_direct_deps']:>3d}  "
          f"transitive={row['n_transitive_deps']:>4d}  "
          f"fraction={row['transitive_fraction']:.3f}")

print()

# Sample layer violations
if len(violations_df) > 0:
    sample = violations_df.sort_values("severity", ascending=False).head(10)
    print("--- Sample layer violations (top 10 by severity) ---")
    for _, row in sample.iterrows():
        print(f"  {row['source']:<40s} -> {row['dependency']:<40s}  "
              f"layer {row['source_layer']} -> {row['dep_layer']}  ({row['violation_type']})")
else:
    print("--- No layer violations ---")

## 8. Persist Output

In [ ]:
# Merge centrality + layers + transitive deps into graph_analysis.parquet
graph_analysis = centrality_df.merge(layer_df, on="cmake_target", how="outer")
graph_analysis = graph_analysis.merge(transitive_df, on="cmake_target", how="outer")

out_path = ds.save_intermediate("graph_analysis", graph_analysis)
print(f"Saved graph_analysis.parquet: {graph_analysis.shape[0]} rows x {graph_analysis.shape[1]} cols")
print(f"  -> {out_path}")